In [ ]:
from src.cgx.parser.parse_codebase import parse_codebase

In [2]:
project_path = "/home/rmohammadi/Downloads/Modal"

chunks, relations = parse_codebase(project_path)
print("Parsed chunks:", len(chunks))
print("Call relations:", len(relations))

Parsed chunks: 78
Call relations: 293


In [3]:
from src.cgx.graph.build_graph import build_knowledge_graph

In [4]:
G = build_knowledge_graph(chunks, relations)

In [5]:
chunks[1]

{'id': '/home/rmohammadi/Downloads/Modal/gemma12b.py::class::Gemma12BIT',
 'type': 'class',
 'name': 'Gemma12BIT',
 'file': '/home/rmohammadi/Downloads/Modal/gemma12b.py',
 'module_path': 'gemma12b',
 'code': 'class Gemma12BIT:\n    """\n    Encapsulates deployment of the Gemma-3-12B-IT model on Modal.\n\n    Responsibilities:\n    - Loads the Hugging Face Gemma model and processor at container startup.\n    - Provides text generation via structured chat inputs.\n    """\n\n    @modal.enter()\n    def load_model(self):\n        """\n        Lifecycle hook for container startup.\n\n        Loads:\n        - Hugging Face AutoProcessor for tokenization and preprocessing.\n        - Gemma3ForConditionalGeneration model, configured with FlashAttention-2.\n\n        Ensures:\n        - Model and processor are cached locally.\n        - Processor has a valid `pad_token_id`.\n        """\n        from transformers import AutoProcessor, Gemma3ForConditionalGeneration\n        import torch\n\n  

In [11]:
len(chunks)

78

In [9]:
from src.cgx.embeddings.records import make_index_records, prepare_embedding_corpus


In [10]:
records = make_index_records(chunks, G)

In [12]:
len(records)

78

In [13]:
records[0]

{'id': '/home/rmohammadi/Downloads/Modal/gemma12b.py',
 'type': 'file',
 'name': 'gemma12b.py',
 'file': '/home/rmohammadi/Downloads/Modal/gemma12b.py',
 'class_name': None,
 'signature': None,
 'docstring': 'gemma12b.py\n\nThis module defines a secure deployment of the `google/gemma-3-12b-it` model \non Modal with GPU acceleration. It sets up a containerized environment \noptimized for FlashAttention, loads the Gemma model, and provides a method \nfor text generation through a chat template.\n\nFeatures:\n- GPU-backed execution using A100-40GB.\n- Secure Hugging Face model download with Modal secrets.\n- Optimized setup with PyTorch, FlashAttention, and Hugging Face Hub caching.\n- Provides inference via `generate_text` using structured chat inputs.',
 'doc_first_sentence': 'gemma12b.py',
 'module_path': 'gemma12b',
 'parent_file_id': '/home/rmohammadi/Downloads/Modal/gemma12b.py',
 'parent_class_id': None,
 'defines_children_ids': [],
 'calls_out_ids': [],
 'calls_out_unresolved': []

In [14]:
which = ("intent", "impl")

In [15]:
corpus = prepare_embedding_corpus(records, which=which)

In [20]:
from typing import Any, Dict, List, Optional, Tuple
import numpy as np
import os
import json

In [21]:
# Split corpus per-view
per_view: Dict[str, List[Dict[str, Any]]] = {"intent": [], "impl": []}
for row in corpus:
    vw = row.get("view")
    if vw in per_view:
        per_view[vw].append(row)

In [25]:
per_view

{'intent': [{'chunk_id': '/home/rmohammadi/Downloads/Modal/gemma12b.py',
   'view': 'intent',
   'text': 'type: file\nsymbol: /home/rmohammadi/Downloads/Modal/gemma12b.py\nname: gemma12b.py\nfile: /home/rmohammadi/Downloads/Modal/gemma12b.py\nclass: \nsignature: \nsummary: gemma12b.py\nimports: \nreads: \nwrites: \nraises: \ncalls_out: \ncalled_by_count: 0\nmetrics: n_loc=154, n_params=, async=False, generator=False',
   'tokens_estimate': 75,
   'type': 'file',
   'name': 'gemma12b.py',
   'file': '/home/rmohammadi/Downloads/Modal/gemma12b.py'},
  {'chunk_id': '/home/rmohammadi/Downloads/Modal/gemma12b.py::class::Gemma12BIT',
   'view': 'intent',
   'text': 'type: class\nsymbol: /home/rmohammadi/Downloads/Modal/gemma12b.py::class::Gemma12BIT\nname: Gemma12BIT\nfile: /home/rmohammadi/Downloads/Modal/gemma12b.py\nclass: \nsignature: \nsummary: Encapsulates deployment of the Gemma-3-12B-IT model on Modal.\nimports: \nreads: \nwrites: \nraises: \ncalls_out: \ncalled_by_count: 0\nmetrics: 

In [39]:
from src.cgx.embeddings.build import build_embeddings

def _to_faiss_ids(rows: List[Dict[str, Any]]) -> np.ndarray:
    """Return a dense 0..N-1 int64 id array for FAISS; keep original ids in meta/rows."""
    return np.arange(len(rows), dtype=np.int64)
metric = "cosine"
model_name = "jinaai/jina-embeddings-v2-base-code"
batch_size = 64

In [41]:
!pip install ipywidgets --upgrade

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.6 MB/s eta 0:00:00a 0:00:01


In [47]:
from src.cgx.embeddings.index import build_faiss_index
index_type = 'flat'

In [48]:
# Build embeddings & FAISS per view
indices: Dict[str, Any] = {"views": {}, "metric": metric}
for view_name, rows in per_view.items():
    if not rows:
        indices["views"][view_name] = {
            "index": None,
            "meta": None,
            "rows": [],
            "ids": np.array([], dtype=np.int64),
        }
        continue

    # Always use build_embeddings with corpus rows
    embs = build_embeddings(
        rows,
        model_name=model_name,
        backend="auto",
        normalize=(metric in {"cosine", "ip"}),
        batch_size=batch_size,
        field_strategy="auto",
    )

    embs = np.asarray(embs, dtype=np.float32)

    # Build FAISS index
    index, meta = build_faiss_index(
        embs,
        metric=metric,
        index=index_type,
        return_meta=True,
    )

    # Preserve original chunk IDs
    orig_ids = [r.get("chunk_id") for r in rows]
    meta = {**(meta or {}), "orig_chunk_ids": orig_ids}

    indices["views"][view_name] = {
        "index": index,
        "meta": meta,
        "rows": rows,
        "ids": _to_faiss_ids(rows),
    }

[INFO] 09:20:11 sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: jinaai/jina-embeddings-v2-base-code
Some weights of BertModel were not initialized from the model checkpoint at jinaai/jina-embeddings-v2-base-code and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.intermediate.dense.bias', 'encoder.layer.1.intermediate.dense.weight', 'encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.1.output.LayerNorm.weight', 'encoder.layer.1.output.dense.bias', 'encoder.layer.1.output.dense.weight', 'encoder.layer.10.intermediate.dense.bias', 'encoder.layer.10.intermediate.dense.weight', 'encoder.layer.10.output.LayerNorm.bias', 'encoder.layer.10.output.LayerNorm.weight', 'encoder.layer.

In [50]:
indices['views']

{'intent': {'index': <faiss.swigfaiss_avx2.IndexIDMap2; proxy of <Swig Object of type 'faiss::IndexIDMap2Template< faiss::Index > *' at 0x74be06ba0720> >,
  'meta': {'dim': 768,
   'metric': 'cosine',
   'faiss_metric': 'IP',
   'index_type': 'flat',
   'nlist': None,
   'nprobe': None,
   'M': None,
   'efConstruction': None,
   'efSearch': None,
   'used_gpu': False,
   'normalized_for_cosine': True,
   'num_vectors': 78,
   'orig_chunk_ids': ['/home/rmohammadi/Downloads/Modal/gemma12b.py',
    '/home/rmohammadi/Downloads/Modal/gemma12b.py::class::Gemma12BIT',
    '/home/rmohammadi/Downloads/Modal/gemma12b.py::method::Gemma12BIT.load_model',
    '/home/rmohammadi/Downloads/Modal/gemma12b.py::method::Gemma12BIT.generate_text',
    '/home/rmohammadi/Downloads/Modal/gptOss20b.py',
    '/home/rmohammadi/Downloads/Modal/gptOss20b.py::class::StopOnToken',
    '/home/rmohammadi/Downloads/Modal/gptOss20b.py::method::StopOnToken.__init__',
    '/home/rmohammadi/Downloads/Modal/gptOss20b.py::m

In [14]:
import json
def load_jsonl(path: str):
    out: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            if isinstance(obj, dict):
                out.append(obj)
    return out


In [17]:
records = load_jsonl("/home/rmohammadi/Downloads/Code_Gen/cgx_index/chunks.jsonl")

In [18]:
records

[{'id': '/home/rmohammadi/Downloads/Code_Gen/src/st_embedder.py',
  'type': 'file',
  'name': 'st_embedder.py',
  'file': '/home/rmohammadi/Downloads/Code_Gen/src/st_embedder.py',
  'module_path': 'st_embedder',
  'code': 'import numpy as np\nfrom sentence_transformers import SentenceTransformer\ndef make_model(): ...\nclass STEmbedder: ...',
  'meta': {'docstring': None,
   'members': {'functions': [{'name': 'make_model',
      'signature': '()',
      'docstring': None}],
    'classes': [{'name': 'STEmbedder',
      'signature': 'class STEmbedder',
      'docstring': None}],
    'imports': ['import numpy as np',
     'from sentence_transformers import SentenceTransformer'],
    'globals': []},
   'metrics': {'n_loc': 19}}},
 {'id': '/home/rmohammadi/Downloads/Code_Gen/src/st_embedder.py::class::STEmbedder',
  'type': 'class',
  'name': 'STEmbedder',
  'file': '/home/rmohammadi/Downloads/Code_Gen/src/st_embedder.py',
  'module_path': 'st_embedder',
  'code': 'class STEmbedder:\n    de